In [1]:
# build_network_by_index() is defined by a function build graph (combine all nodes and edges)
# Nodes file is ...nodes.csv files where first column is node and second column is its "direction" attribute
# Each column is not necesssarily named that
# Edges file is ...edges.csv files where first column is source and second column is target node (undirected graph)
# and third column is "direction" attribute
# Each column is not necesssarily named that


In [3]:
# define layer 0

In [205]:
import pandas as pd
import networkx as nx

def build_network_by_index(
    B_nodes_csv,
    AA_edges_csv,
    AB_edges_csv,
    has_header=True
):
    header = 0 if has_header else None
    G = nx.Graph()

    # Add Nodes
    nodes = pd.read_csv(B_nodes_csv, header=header)
    for i, row in nodes.iterrows():
        G.add_node(row.iloc[0], direction=row.iloc[1])

    # Add Eeges
    for edge_csv in [AA_edges_csv, AB_edges_csv]:
        edges = pd.read_csv(edge_csv, header=header)
        for i, row in edges.iterrows():
            G.add_edge(row.iloc[0],row.iloc[1],direction=row.iloc[2])

# --------------------------------------------------------------------------------------------------
    # Ensure every node has a direction (nodes created via edges may not)
    #for n in G.nodes:
        #if "direction" in G.nodes[n]:
            #print(n, G.nodes[n]["direction"])

        #if "direction" not in G.nodes[n]:
        #    G.nodes[n]["direction"] = 0 
# --------------------------------------------------------------------------------------------------

    
    # Define layer 0
    layer_0 = []
    for n in nodes.iloc[:, 0]:
        if G.degree(n) > 0:
            layer_0.append(n)
    
    return G,layer_0


In [207]:
# layer 1 is all nodes neighbor to nodes in layer 0, that are not in layer 0
# in fact layer d is all nodes with distance d away from layer 0

In [209]:
def build_layers(G, layer_0, max_depth=None):
    # list of sets --> layers[d] will contain all nodes at distance d (starting will change later)
    layers = []

    
    layers.append(set(layer_0))    # "layer 0" is the source nodes
    visited = set(layer_0)    # "visited" is all nodes we have already visited
    current_layer = set(layer_0)    # nodes at the current distance from layer 0
    depth = 0    # current distance from layer 0

    
    while current_layer: #until no new nodes are found
        
        if max_depth is not None and depth >= max_depth:    # until max depth (DO NOT REMOVE KEPT FOR DEBUGGING)
            break

        next_layer = set()  # next_layer nodes = distance (depth + 1)

        for node in current_layer:    # for every node in the current layer
            for neighbor in G.neighbors(node):     # look at all neighbors of this node
                if neighbor not in visited:     # add neighbor only if it hasn't appeared in any earlier layer
                    next_layer.add(neighbor)

        if not next_layer:     # if no new nodes were found, we are done
            break

        layers.append(next_layer)  # Save next layer

        visited.update(next_layer)  # Updated all visited layers (already discovered and assigned a layer)

        current_layer = next_layer  # Current layer is the next layer
        depth = depth + 1  # Increment depth

    return layers  # return list of layers: layers[d]


In [211]:
# For each node in layer 1(d) count all edges between layer 1(d) and layer 0(d-1) --> (where d represents layer)
# For each node in layer 1(d), create two variables called pos_votes and neg_votes. 

# deletes edges based on previous layer
def filter_level(G, curr_layer, prev_layer):
    temp = 0 # DELETE LATER-----------------------
    for v in curr_layer:  # for every node v in the current layer
        
        pos_votes = 0
        neg_votes = 0
        pos_edges = []   # edges that voted positive
        neg_edges = []   # edges that voted negative

        for u in list(G.neighbors(v)):  # u is neighbors of v from previous layer
            if u in prev_layer:

                node_dir = float(G.nodes[u]["direction"])
                edge_dir = float(G[u][v]["direction"])
                vote = node_dir * edge_dir

                if vote > 0:
                    pos_votes += 1
                    pos_edges.append((u, v))
                else:
                    neg_votes += 1
                    neg_edges.append((u, v))

        # if no votes, skip (otherwise dividing by zero)
        total = pos_votes + neg_votes
        if total == 0:
            continue

        # set direction of v based on majority vote
        if pos_votes > neg_votes:
            G.nodes[v]["direction"] = 1
            minority_edges = neg_edges
        else:
            G.nodes[v]["direction"] = -1
            minority_edges = pos_edges

        # calculate puc_value --> minority / total
        G.nodes[v]["puc_value"] = len(minority_edges) / total

        if G.nodes[v]["puc_value"] > 0.2:
            # delete all edges between v and nodes in the previous layer
            for u in list(G.neighbors(v)):
                if u in prev_layer:
                    G.remove_edge(u, v)
        else:
            # delete only minority-vote edges
            for a, b in minority_edges:
                G.remove_edge(a, b)
    return G


# deletes edges in same level
def same_level_edges(G, curr_layer):
    
    for u in curr_layer:   # only consider edges where BOTH endpoints are in the same layer
        for v in list(G.neighbors(u)):
            if v in curr_layer:
                
                node_prod = float(G.nodes[u]["direction"]) * float(G.nodes[v]["direction"])
                edge_dir = float(G[u][v]["direction"])

                # if product of node directions != edge direction, delete edge
                if node_prod != edge_dir:
                    if G.has_edge(u, v):  # avoid double-delete since undirected
                        G.remove_edge(u, v)

    return G

In [227]:
import pandas as pd

#puc_delete_threshold=0.2
graph_A, layer_0 = build_network_by_index("cpx_node.csv","new_pls-pls_edges.csv","new_pls-cpx_edges.csv",has_header=True)
layers = build_layers(graph_A, layer_0, max_depth=None)
print(graph_A)

d = 1 #Start from distance = 1
while d < len(layers):
    
    # ------------------------------------------------------------------------------------------------------------
    print("Total layers:", len(layers))
    # Number of nodes per layer
    sum = 0
    for i, layer in enumerate(layers):
        if i>0:
            sum = sum + len(layer)
            #print(f"Layer {i}: {len(layer)} nodes")
    print("Total number of nodes kept",sum)
    # ------------------------------------------------------------------------------------------------------------

    print("Starting with:",d)
    prev_layer = set(layers[d - 1])   # define previous layer
    # ------------------------------------------------------------------------------------------------------------
    #for u in prev_layer:
    #    print(u, "->", graph_A.nodes[u].get("direction"))
    # ------------------------------------------------------------------------------------------------------------
    curr_layer = layers[d]   # define current layer
    
    # ------------------------------------------------------------------------------------------------------------
    #used_u = {u for v in curr_layer for u in graph_A.neighbors(v) if u in prev_layer}
    #print("Not used:", prev_layer - used_u)  
    #missing_dir = [u for u in prev_layer if "direction" not in graph_A.nodes[u]]
    #print("Prev_layer nodes missing direction:", len(missing_dir))
    #print(missing_dir)
    # ------------------------------------------------------------------------------------------------------------

    graph_A = filter_level(graph_A, curr_layer, prev_layer)
    graph_A = same_level_edges(graph_A, curr_layer)
    layers = build_layers(graph_A, layer_0, max_depth=None)
    print("done with layer:",d)
    d += 1



rows = []
for d in range(1, len(layers)):
    for node in layers[d]:
        rows.append({
            "node": node,
            "direction": graph_A.nodes[node].get("direction"),
            "puc_value": graph_A.nodes[node].get("puc_value")
        })
print(len(rows))
df = pd.DataFrame(rows, columns=["node", "direction", "puc_value"])
df.to_csv("all.csv", index=False)


Graph with 1332 nodes and 9512 edges
Total layers: 3
Total number of nodes kept 529
Starting with: 1
done with layer: 1
Total layers: 4
Total number of nodes kept 529
Starting with: 2
done with layer: 2
Total layers: 5
Total number of nodes kept 529
Starting with: 3
done with layer: 3
Total layers: 6
Total number of nodes kept 527
Starting with: 4
done with layer: 4
Total layers: 6
Total number of nodes kept 527
Starting with: 5
done with layer: 5
527
